[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/SFM/blob/main/Quantlets/Ch_03/SFM_ch3_densities/SFM_ch3_densities.ipynb)

# SFM_ch3_densities

Densities of alpha-stable laws computed by FFT inversion of the characteristic function (Nolan S0 and S1 parameterisations): the effect of alpha (tails and peak, linear and log scale), of beta (skewness) and of gamma (scale); S1 vs S0 near alpha = 1 with beta = 0.8; the three closed-form cases Normal, Cauchy and Levy.

*Statistics of Financial Markets (SFM), Chapter 3: alpha-stable distributions. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/SFM/Chapter_3'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import json
import urllib.request
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgb
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

## Chart style

In [ ]:
# Palette (RGB values of latex/preamble.tex)
MainBlue = '#1A3A6E'
IDAred = '#CD0000'
Forest = '#2E7D32'
Amber = '#B5853F'
Orange = '#E67E22'
Purple = '#8E44AD'
Teal = '#17A2B8'
Crimson = '#DC3545'
DarkText = '#1F2A44'          # text, axes and ticks (dark navy, not grey)

PALETTE = [MainBlue, IDAred, Forest, Amber, Purple, Orange, Teal, Crimson]
# fixed colours for the series used in several chapters
COL = {'sp500': MainBlue, 'bet': IDAred, 'bettr': Orange, 'dax': Teal, 'btc': Amber, 'eth': Crimson,
       'eurron': Forest, 'gold': Purple, 'vix': Crimson, 'stoxx50': Forest, 'ndx': Teal}


def apply():
    """Set the course style for matplotlib."""
    rc = plt.rcParams
    rc['figure.facecolor'] = 'none'
    rc['axes.facecolor'] = 'none'
    rc['savefig.facecolor'] = 'none'
    rc['savefig.transparent'] = True
    rc['axes.grid'] = False
    rc['font.family'] = 'sans-serif'
    rc['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
    # font sizes for slides (charts are 9-11 inches wide and shown at 8-14 cm)
    rc['font.size'] = 12
    rc['axes.labelsize'] = 13
    rc['axes.titlesize'] = 13
    rc['xtick.labelsize'] = 11.5
    rc['ytick.labelsize'] = 11.5
    rc['legend.fontsize'] = 11
    rc['axes.spines.top'] = False
    rc['axes.spines.right'] = False
    rc['axes.linewidth'] = 0.6
    rc['lines.linewidth'] = 1.4
    rc['legend.facecolor'] = 'none'
    rc['legend.framealpha'] = 0
    rc['legend.frameon'] = False
    for k in ('text.color', 'axes.labelcolor', 'axes.edgecolor', 'xtick.color', 'ytick.color', 'axes.titlecolor'):
        rc[k] = DarkText
    rc['axes.prop_cycle'] = mpl.cycler(color=PALETTE)


def legend_outside_bottom(ax, ncol=2, y=-0.22, **kw):
    """Place the legend outside the plot, bottom centre (for a figure with several axes, pass the last one
    or use fig.legend with the same arguments)."""
    return ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False, **kw)


def fig_legend_bottom(fig, handles=None, labels=None, ncol=3, y=-0.02):
    """One legend for a whole figure (several panels), below the panels."""
    if handles is None:
        handles, labels = [], []
        for ax in fig.axes:
            for h, l in zip(*ax.get_legend_handles_labels()):
                if l not in labels and not l.startswith('_'):
                    handles.append(h)
                    labels.append(l)
    return fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def _is_grey(c, tol=0.06):
    try:
        r, g, b = to_rgb(c)
    except ValueError:
        return False
    return max(r, g, b) - min(r, g, b) < tol and 0.25 < (r + g + b) / 3 < 0.95


def check_no_grey(fig):
    """House rule: no grey series and no grey text. Raises ValueError listing the offending elements."""
    bad = []
    for ax in fig.axes:
        for ln in ax.get_lines():
            if not ln.get_label().startswith('_') and _is_grey(ln.get_color()):
                bad.append(f'line {ln.get_label()!r}')
        for coll in ax.collections:
            fc = coll.get_facecolor()
            if len(fc) and coll.get_label() and not coll.get_label().startswith('_') and _is_grey(fc[0][:3]):
                bad.append(f'series {coll.get_label()!r}')
        for t in ax.texts + [ax.title, ax.xaxis.label, ax.yaxis.label]:
            if t.get_text() and _is_grey(t.get_color()):
                bad.append(f'text {t.get_text()[:30]!r}')
    if bad:
        raise ValueError('grey elements: ' + ', '.join(bad))


def save_fig(name, out_dir=None, show=True):
    """Save the figure as transparent PDF and PNG (OUT_DIR if defined, else the current folder), then show it."""
    d = out_dir or globals().get('OUT_DIR', '.')
    plt.savefig(os.path.join(d, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(d, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    if show:
        plt.show()
    plt.close()


apply()
# the chapter scripts call the style as st.<name> (import sfm_style as st): the same names here
import types
st = types.SimpleNamespace(COL=COL, PALETTE=PALETTE, apply=apply, legend_outside_bottom=legend_outside_bottom,
                           fig_legend_bottom=fig_legend_bottom, save_fig=save_fig, check_no_grey=check_no_grey)

## Data

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/SFM/main/data/market/'
# local copy of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
END = '2026-09-18'          # last day of the saved data

# name -> (symbol, label, group, field, default start)
SERIES = {
    # equity indices
    'sp500':    ('GSPC.INDX',     'S&P 500',               'Equity', 'close', '2000-01-01'),
    'ndx':      ('NDX.INDX',      'Nasdaq 100',            'Equity', 'close', '2000-01-01'),
    'dax':      ('GDAXI.INDX',    'DAX',                   'Equity', 'close', '2000-01-01'),
    'stoxx50':  ('STOXX50E.INDX', 'Euro Stoxx 50',         'Equity', 'close', '2000-01-01'),
    'nikkei':   ('N225.INDX',     'Nikkei 225',            'Equity', 'close', '2000-01-01'),
    'bet':      ('BET',           'BET',                   'Equity', 'close', '2000-01-01'),
    'bettr':    ('BETTR.INDX',    'BET-TR',                'Equity', 'close', '2014-09-23'),
    'betxt':    ('BETXT.INDX',    'BET-XT',                'Equity', 'close', '2011-08-12'),
    'betfi':    ('BETFI.INDX',    'BET-FI',                'Equity', 'close', '2012-01-25'),
    'wig20':    ('WIG20.INDX',    'WIG20',                 'Equity', 'close', '2000-01-01'),
    'bux':      ('BUX.INDX',      'BUX',                   'Equity', 'close', '2000-01-01'),
    'px':       ('PX.INDX',       'PX',                    'Equity', 'close', '2000-01-01'),
    'vix':      ('VIX.INDX',      'VIX',                   'Volatility', 'close', '2000-01-01'),
    # ETFs (adjusted close)
    'spy':      ('SPY.US',        'SPY',                   'ETF', 'adjusted_close', '2000-01-01'),
    'tlt':      ('TLT.US',        'TLT (US Treasuries 20y+)', 'ETF', 'adjusted_close', '2002-07-30'),
    'gld':      ('GLD.US',        'GLD (gold ETF)',        'ETF', 'adjusted_close', '2004-11-18'),
    'tvbetetf': ('TVBETETF.RO',   'TVBETETF',              'ETF', 'adjusted_close', '2015-06-12'),
    # Bucharest Stock Exchange (adjusted close)
    'tlv':      ('TLV.RO',        'Banca Transilvania',    'Stock', 'adjusted_close', '2000-01-01'),
    'snp':      ('SNP.RO',        'OMV Petrom',            'Stock', 'adjusted_close', '2001-09-04'),
    'brd':      ('BRD.RO',        'BRD',                   'Stock', 'adjusted_close', '2001-01-16'),
    'tgn':      ('TGN.RO',        'Transgaz',              'Stock', 'adjusted_close', '2008-01-25'),
    'sng':      ('SNG.RO',        'Romgaz',                'Stock', 'adjusted_close', '2013-11-12'),
    'snn':      ('SNN.RO',        'Nuclearelectrica',      'Stock', 'adjusted_close', '2013-11-04'),
    'h2o':      ('H2O.RO',        'Hidroelectrica',        'Stock', 'adjusted_close', '2023-07-12'),
    # US stocks (adjusted close)
    'aapl':     ('AAPL.US',       'Apple',                 'Stock', 'adjusted_close', '2000-01-01'),
    'msft':     ('MSFT.US',       'Microsoft',             'Stock', 'adjusted_close', '2000-01-01'),
    'nvda':     ('NVDA.US',       'NVIDIA',                'Stock', 'adjusted_close', '2000-01-01'),
    'jpm':      ('JPM.US',        'JPMorgan Chase',        'Stock', 'adjusted_close', '2000-01-01'),
    # crypto (close, 7 days a week)
    'btc':      ('BTC-USD.CC',    'Bitcoin',               'Crypto', 'close', '2014-09-17'),
    'eth':      ('ETH-USD.CC',    'Ethereum',              'Crypto', 'close', '2015-08-07'),
    'sol':      ('SOL-USD.CC',    'Solana',                'Crypto', 'close', '2020-04-11'),
    'usdt':     ('USDT-USD.CC',   'Tether (USDT)',         'Crypto', 'close', '2015-02-26'),
    # FX and commodities
    'eurron':   ('REF:EUR',       'EUR/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'usdron':   ('REF:USD',       'USD/RON (BNR reference)', 'FX', 'close', '2005-07-01'),
    'eurron_eodhd': ('EURRON.FOREX', 'EUR/RON (EODHD)',    'FX', 'close', '2005-07-01'),
    'eurusd':   ('EURUSD.FOREX',  'EUR/USD',               'FX', 'close', '2002-05-06'),
    'gold':     ('XAUUSD.FOREX',  'Gold (XAU/USD)',        'Commodity', 'close', '2000-01-01'),
    # government bond yields (close, % p.a.)
    'us10y':    ('US10Y.GBOND',   'US 10Y yield',          'Yield', 'close', '2000-01-01'),
    'de10y':    ('DE10Y.GBOND',   'Germany 10Y yield',     'Yield', 'close', '2007-03-28'),
    'ro10y':    ('RO10Y.GBOND',   'Romania 10Y yield',     'Yield', 'close', '2007-08-17'),
}
LABELS = {k: v[1] for k, v in SERIES.items()}
_CACHE = {}


def read_market(symbol):
    """Daily table of one symbol: local copy of the course data, otherwise the SFM repository on GitHub."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname) if MARKET_DIR else ''
    src = path if path and os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def read_reference_rate(currency='EUR', start='2005-07-01', end=END):
    """Official BNR reference rate (RON per unit of currency), from the public yearly XML archives."""
    key = (currency, start, end)
    if key in _CACHE:
        return _CACHE[key]
    rows = []
    for y in range(max(int(start[:4]), 2005), int(end[:4]) + 1):
        url = f'https://curs.bnr.ro/files/xml/years/nbrfxrates{y}.xml'
        xml = urllib.request.urlopen(urllib.request.Request(url, headers={'User-Agent': 'Mozilla'}),
                                     timeout=60).read().decode()
        for d, body in re.findall(r'<Cube date="([\d-]+)">(.*?)</Cube>', xml, re.S):
            m = re.search(rf'<Rate currency="{currency}"(?: multiplier="\d+")?>([\d.]+)</Rate>', body)
            if m:
                rows.append((d, float(m.group(1))))
    s = pd.DataFrame(rows, columns=['date', 'close']).drop_duplicates('date').set_index('date')['close']
    s.index = pd.to_datetime(s.index)
    _CACHE[key] = s.sort_index().loc[start:end]
    return _CACHE[key]


def load_close(name, start=None, end=END, field=None):
    """Daily closing price of a named series, cleaned with the course conventions."""
    symbol, label, group, field0, start0 = SERIES[name]
    start, field = start or start0, field or field0
    if symbol.startswith('REF:'):
        return read_reference_rate(symbol.split(':')[1], start=start, end=end).rename(name)
    t = read_market(symbol)
    s = (t[field] if field in t.columns else t['close']).loc[start:end]
    s = pd.to_numeric(s, errors='coerce').dropna()
    if group != 'Yield':
        s = s[s > 0]
    if group != 'Crypto':
        s = s[s.index.dayofweek < 5]          # no weekend quotes
        s = s[s.diff() != 0]                  # no holidays filled with the previous price
    s.index.name = 'date'
    return s.rename(name)


def load_ohlc(name, start=None, end=END):
    """Open, high, low, close (and volume) of a series with OHLC data (range-based volatility estimators)."""
    symbol, _, group, _, start0 = SERIES[name]
    t = read_market(symbol).loc[start or start0:end]
    cols = [c for c in ('open', 'high', 'low', 'close', 'volume') if c in t.columns]
    t = t[cols].dropna()
    if group != 'Crypto':
        t = t[t.index.dayofweek < 5]
    return t[(t[['open', 'high', 'low', 'close']] > 0).all(axis=1)] if 'open' in cols else t


def log_returns(name, start=None, end=END, pct=True):
    """Daily log returns r_t = ln P_t - ln P_{t-1} (in % by default), on the series' own calendar."""
    r = np.log(load_close(name, start, end)).diff().dropna()
    return (100 * r if pct else r).rename(name)


def simple_returns(name, start=None, end=END, pct=True):
    """Daily simple returns R_t = P_t / P_{t-1} - 1 (in % by default)."""
    r = load_close(name, start, end).pct_change().dropna()
    return (100 * r if pct else r).rename(name)


def load_panel(names, start=None, end=END, kind='close'):
    """Several series aligned on date: kind = 'close' or 'returns' (NaN where a market does not trade)."""
    f = load_close if kind == 'close' else log_returns
    return pd.concat([f(n, start, end) for n in names], axis=1)


def periods_per_year(r):
    """Average number of observations per calendar year (the actual frequency of the series)."""
    years = (r.index[-1] - r.index[0]).days / 365.25
    return len(r) / years

## Analysis

In [ ]:
START = '2000-01-01'

ASSETS = ['bet', 'sp500', 'dax', 'btc']

COLORS = {'sp500': '#1A3A6E', 'dax': '#17A2B8', 'bet': '#CD0000', 'btc': '#B5853F'}

MODEL_COL = {'Normal': '#CD0000', 'Student-t': '#2E7D32', 'Stable': '#1A3A6E', 'Data': '#B5853F'}

SEED = 2026

def stable_cf(t, alpha, beta, gamma=1.0, delta=0.0, param=0):
    """Characteristic function E exp(itX) of S(alpha, beta, gamma, delta; param), Nolan's S0 (param=0) or S1."""
    t = np.asarray(t, float)
    a = np.abs(gamma * t)
    if alpha != 1:
        tan = np.tan(np.pi * alpha / 2)
        if param == 0:      # -|gt|^a [1 + i b tan(pi a/2) sign(t) (|gt|^(1-a) - 1)], written without 0^(1-a)
            psi = -a ** alpha - 1j * beta * tan * (gamma * t - np.sign(t) * a ** alpha)
        else:               # -|gt|^a [1 - i b tan(pi a/2) sign(t)]
            psi = -a ** alpha * (1 - 1j * beta * tan * np.sign(t))
    else:
        arg = a if param == 0 else np.abs(t)
        lg = np.log(np.where(arg > 0, arg, 1.0))
        psi = -a * (1 + 1j * beta * (2 / np.pi) * np.sign(t) * lg)
    return np.exp(psi + 1j * delta * t)


def s1_to_s0(alpha, beta, gamma, delta1):
    """Location in S0 from the location in S1 (Nolan, 2020): only delta changes."""
    if alpha != 1:
        return delta1 + beta * gamma * np.tan(np.pi * alpha / 2)
    return delta1 + beta * (2 / np.pi) * gamma * np.log(gamma)


def s0_to_s1(alpha, beta, gamma, delta0):
    """Location in S1 from the location in S0."""
    if alpha != 1:
        return delta0 - beta * gamma * np.tan(np.pi * alpha / 2)
    return delta0 - beta * (2 / np.pi) * gamma * np.log(gamma)


class StableGrid:
    """Density and distribution function of the standard S0(alpha, beta, 1, 0) law on a fine grid, by FFT
    inversion of the characteristic function, f(x) = (1/2pi) int phi(t) exp(-itx) dt (Mittnik et al., 1999).
    Grid step h = 0.01 on [-327.68, 327.67]; the power-law tail approximation is used beyond the grid."""

    def __init__(self, alpha, beta, h=0.01, N=2 ** 16):
        k = np.fft.fftfreq(N, d=1.0 / N)                        # 0, 1, ..., N/2-1, -N/2, ..., -1
        dt = 2 * np.pi / (N * h)
        phi = stable_cf(k * dt, alpha, beta, 1.0, 0.0, 0) * np.where(k % 2 == 0, 1.0, -1.0)
        f = np.real(np.fft.fft(phi)) * dt / (2 * np.pi)
        self.alpha, self.beta, self.h = alpha, beta, h
        self.x = (np.arange(N) - N / 2) * h
        self.f = np.maximum(f, 1e-300)
        self.logf = np.log(self.f)
        # tail constant: P(X > x) ~ c_alpha (1 + beta) x^(-alpha), c_alpha = Gamma(alpha) sin(pi alpha / 2) / pi
        from scipy.special import gamma as G
        self.c = 0.0 if alpha >= 2 else G(alpha) * np.sin(np.pi * alpha / 2) / np.pi
        left = self.c * (1 - beta) * abs(self.x[0]) ** (-alpha) if alpha < 2 else 0.0
        right = self.c * (1 + beta) * self.x[-1] ** (-alpha) if alpha < 2 else 0.0
        cells = 0.5 * (self.f[1:] + self.f[:-1]) * h
        self.F = np.clip(left + np.concatenate([[0.0], np.cumsum(cells)]), 0.0, 1.0)           # P(X <= x), from the left
        self.S = np.clip(right + np.concatenate([np.cumsum(cells[::-1])[::-1], [0.0]]), 0.0, 1.0)  # P(X > x), from the right

    def pdf(self, z):
        z = np.asarray(z, float)
        out = np.exp(np.interp(z, self.x, self.logf))
        far = np.abs(z) > self.x[-1]
        if far.any() and self.alpha < 2:
            out[far] = self.alpha * self.c * (1 + np.sign(z[far]) * self.beta) * np.abs(z[far]) ** (-self.alpha - 1)
        return out

    def cdf(self, z):
        z = np.asarray(z, float)
        return np.where(z <= 0, np.interp(z, self.x, self.F), 1 - np.interp(z, self.x, self.S))

    def sf(self, z):
        return 1 - self.cdf(z) if np.ndim(z) == 0 else np.where(np.asarray(z) > 0, np.interp(z, self.x, self.S), 1 - self.cdf(z))

    def ppf(self, p):
        i = np.searchsorted(self.F, 1e-12) if self.F[0] < 1e-12 else 0
        return np.interp(p, self.F[i:], self.x[i:])


def stable_pdf(x, alpha, beta, gamma=1.0, delta=0.0, param=0):
    """Density of S(alpha, beta, gamma, delta; param) on any points x (FFT grid, see StableGrid)."""
    d0 = delta if param == 0 else s1_to_s0(alpha, beta, gamma, delta)
    return StableGrid(alpha, beta).pdf((np.asarray(x, float) - d0) / gamma) / gamma


def fig_density_alpha(save=True):
    """Standard symmetric stable densities S0(alpha, 0, 1, 0) for several alpha, linear and log scale."""
    x = np.linspace(-8, 8, 801)
    alphas = [2.0, 1.7, 1.5, 1.0, 0.7]
    cols = ['#1A3A6E', '#17A2B8', '#2E7D32', '#E67E22', '#CD0000']
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.9))
    out = {}
    for a, c in zip(alphas, cols):
        f = StableGrid(a, 0.0).pdf(x) if a >= 1 else stats.levy_stable.pdf(x, a, 0.0)
        lab = f'alpha = {a:g}' + (' (Normal, variance 2)' if a == 2 else ' (Cauchy)' if a == 1 else '')
        axes[0].plot(x, f, color=c, label=lab)
        axes[1].semilogy(x, f, color=c, label='_')
        out[str(a)] = {'f0': float(np.interp(0, x, f)), 'f5': float(np.interp(5, x, f))}
    axes[0].set_xlim(-5, 5)
    axes[0].set_title('Density')
    axes[1].set_title('Density, log scale')
    for ax in axes:
        ax.set_xlabel('x')
    st.fig_legend_bottom(fig, ncol=3, y=0.0)
    fig.tight_layout(rect=(0, 0.03, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch3_density_alpha')
    return out


def fig_density_beta(save=True):
    """The skewness parameter beta (alpha = 1.3) and the scale gamma (alpha = 1.7), S0 parameterisation."""
    x = np.linspace(-6, 6, 801)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.9))
    for b, c in zip([-1, -0.5, 0, 0.5, 1], ['#CD0000', '#E67E22', '#1A3A6E', '#17A2B8', '#2E7D32']):
        axes[0].plot(x, StableGrid(1.3, b).pdf(x), color=c, label=f'beta = {b:g}')
    for g, c in zip([0.5, 1.0, 2.0], ['#8E44AD', '#1A3A6E', '#B5853F']):
        axes[1].plot(x, StableGrid(1.7, 0.0).pdf(x / g) / g, color=c, label=f'gamma = {g:g}')
    axes[0].set_title('alpha = 1.3, gamma = 1: the effect of beta')
    axes[1].set_title('alpha = 1.7, beta = 0: the effect of gamma')
    for ax in axes:
        ax.set_xlabel('x')
        st.legend_outside_bottom(ax, ncol=3, y=-0.18)
    fig.tight_layout()
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch3_density_beta')


def fig_s0_s1(save=True):
    """S1 vs S0 with beta = 0.8: in S1 the density runs away as alpha crosses 1; in S0 it moves smoothly."""
    x = np.linspace(-6, 8, 701)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.9))
    alphas = [0.8, 0.95, 1.05, 1.2]
    cols = ['#CD0000', '#E67E22', '#17A2B8', '#1A3A6E']
    modes = {}
    for k, (ax, par) in enumerate(zip(axes, ['S1', 'S0'])):
        stats.levy_stable.parameterization = par
        for a, c in zip(alphas, cols):
            f = stats.levy_stable.pdf(x, a, 0.8)
            ax.plot(x, f, color=c, label=f'alpha = {a:g}' if k == 0 else '_')
            modes[f'{par}_{a}'] = float(x[np.argmax(f)])
        ax.set_title(f'{par}: beta = 0.8, gamma = 1, delta = 0')
        ax.set_xlabel('x')
    stats.levy_stable.parameterization = 'S1'
    st.fig_legend_bottom(fig, ncol=4, y=0.0)
    fig.tight_layout(rect=(0, 0.08, 1, 1))
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch3_s0_s1')
    return modes


def fig_special_cases(save=True):
    """The three stable laws with a closed-form density: Normal, Cauchy and Levy."""
    x = np.linspace(-5, 8, 1301)
    xl = x[x > 0]
    fig, ax = plt.subplots(figsize=(10, 3.8))
    ax.plot(x, stats.norm.pdf(x, 0, 1), color='#1A3A6E', label='Normal: alpha = 2, gamma = 1/sqrt(2) (variance 1)')
    ax.plot(x, stats.cauchy.pdf(x), color='#E67E22', label='Cauchy: alpha = 1, beta = 0, gamma = 1')
    ax.plot(xl, stats.levy.pdf(xl), color='#2E7D32', label='Levy: alpha = 1/2, beta = 1, gamma = 1 (S1)')
    ax.set_xlabel('x')
    ax.set_ylabel('Density')
    st.legend_outside_bottom(ax, ncol=1, y=-0.15)
    st.check_no_grey(fig)
    if save:
        st.save_fig('sfm_ch3_special_cases')

## Run

In [ ]:
print(fig_density_alpha())
fig_density_beta()
print(fig_s0_s1())
fig_special_cases()

![sfm_ch3_density_alpha](./sfm_ch3_density_alpha.png)

Output: `sfm_ch3_density_alpha.pdf`

![sfm_ch3_density_beta](./sfm_ch3_density_beta.png)

Output: `sfm_ch3_density_beta.pdf`

![sfm_ch3_s0_s1](./sfm_ch3_s0_s1.png)

Output: `sfm_ch3_s0_s1.pdf`

![sfm_ch3_special_cases](./sfm_ch3_special_cases.png)

Output: `sfm_ch3_special_cases.pdf`